<a href="https://colab.research.google.com/github/SolKacil/matematicas-para-ia/blob/main/python/01-algebra-lineal/00_propiedades_matrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 00 &middot; Propiedades de matrices: suma, multiplicación, identidad y transpuesta

**Módulo 1 — Álgebra lineal y geometría diferencial**

Este notebook acompaña al documento `Matematicas_para_IA_00.pdf`, basado en Deisenroth, Faisal y
Ong (2020), *Mathematics for Machine Learning*, sección 2.2.

Las cuatro operaciones que se definen aquí —suma, producto por escalar, producto matricial y
transposición— constituyen el repertorio completo con el que se expresa cualquier modelo lineal, y
por extensión cualquier red neuronal. La propagación hacia adelante de una capa densa es
$\mathbf{y} = W\mathbf{x} + \mathbf{b}$; la propagación hacia atrás introduce $W^\top$; una
conexión residual suma la identidad; el promediado de modelos es una suma de matrices. Ninguna de
esas expresiones contiene nada que no esté en este documento.

El énfasis del notebook está en dos puntos donde la intuición aritmética falla y el código lo hace
evidente: el producto matricial **no es conmutativo**, y el operador `*` de NumPy **no es** el
producto matricial. Ambas confusiones producen código que se ejecuta sin error y devuelve resultados
incorrectos, que es la peor clase de defecto.

## Al terminar será posible

- Enunciar las condiciones de **compatibilidad de formas** de cada operación y anticipar la forma
  del resultado antes de ejecutar el código.
- Implementar el **producto matricial** a partir de su definición y verificarlo contra `@`.
- Distinguir el producto matricial del **producto de Hadamard** y justificar por qué `*` y `@`
  producen resultados distintos.
- Verificar computacionalmente las propiedades **asociativa**, **distributiva** y del **elemento
  neutro**, y construir contraejemplos para la conmutatividad.
- Emplear la **transpuesta**, la regla $(AB)^\top = B^\top A^\top$ y el concepto de **matriz
  simétrica**.

## Qué se da por sabido

Aritmética elemental y lectura de código de Python. Éste es el primer notebook del módulo; no
requiere ninguno de los demás.

## Cómo usar este notebook

1. Con el botón **Open in Colab** no se requiere instalación alguna.
2. Las celdas se ejecutan en orden con `Shift + Enter`.
3. Conviene **anticipar la forma del resultado antes de ejecutar** cada celda y contrastarla con la
   salida: la mayor parte de los errores en código de álgebra lineal son incompatibilidades de
   forma, no errores aritméticos.
4. La sección final, **Tu turno**, contiene los ejercicios propuestos del documento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (5.5, 5.5)

print("numpy:", np.__version__)

---

## 1. Qué es una matriz

Una matriz es un arreglo rectangular de números dispuesto en filas y columnas. Si tiene $m$ filas y
$n$ columnas se dice que es de tamaño $m \times n$, y se escribe $A \in \mathbb{R}^{m \times n}$. La
entrada de la fila $i$ y la columna $j$ se denota $a_{ij}$.

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix} \in \mathbb{R}^{2 \times 3}$$

En NumPy una matriz es un arreglo de dos dimensiones y su tamaño se consulta con el atributo
`.shape`, que devuelve la tupla $(m, n)$ en ese orden: **primero filas, después columnas**. Conviene
adoptar desde el principio la costumbre de verificar formas, porque es la comprobación que detecta
la mayoría de los errores.

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])

print("A =\n", A)
print("\nforma (m, n):", A.shape, " -> 2 filas, 3 columnas")
print("numero de entradas:", A.size)

print("\na[0,0] =", A[0, 0], "   a[1,2] =", A[1, 2], "   (los indices empiezan en 0)")
print("fila 0    =", A[0])          # o A[0, :]
print("columna 1 =", A[:, 1])       # los dos puntos recorren todas las filas

# Un vector no es lo mismo que una matriz de una fila: la forma lo distingue.
v = np.array([1.0, 2.0, 3.0])
print("\nv.shape =", v.shape, "  (vector)")
print("A[0:1].shape =", A[0:1].shape, "  (matriz de una fila)")

---

## 2. Suma de matrices

La suma se define **entrada por entrada**, y sólo está definida entre matrices del **mismo tamaño**:

$$(A + B)_{ij} = a_{ij} + b_{ij}$$

Para $A, B, C$ del mismo tamaño se cumple:

- **Conmutativa:** $A + B = B + A$.
- **Asociativa:** $(A + B) + C = A + (B + C)$.

Ambas son consecuencia inmediata de que la suma de números reales las cumple, y de que la operación
actúa posición por posición. Una matriz $3 \times 2$ y una $2 \times 3$ no se pueden sumar aunque
contengan los mismos seis números: la forma es parte de la definición del objeto.

In [ ]:
A2 = np.array([[1.0, 2.0],
               [3.0, 4.0]])
B2 = np.array([[5.0, 0.0],
               [1.0, 2.0]])

print("A + B =\n", A2 + B2)
print("\nconmutativa  A+B == B+A :", np.allclose(A2 + B2, B2 + A2))

C2 = np.array([[2.0, -1.0], [0.0, 3.0]])
print("asociativa   (A+B)+C == A+(B+C):", np.allclose((A2 + B2) + C2, A2 + (B2 + C2)))

# Formas distintas: la operacion no esta definida.
try:
    A + A.T                          # (2,3) + (3,2)
except ValueError as e:
    print("\nsumar (2,3) con (3,2) ->", e)

> **Advertencia sobre `broadcasting`.** NumPy admite sumar una matriz $m \times n$ con un vector de
> $n$ componentes, repitiendo el vector en cada fila. Es una extensión de la biblioteca, **no** una
> operación del álgebra lineal: la suma matricial exige formas idénticas. La extensión es útil —así
> se añade el vector de sesgo a todas las muestras de un lote— pero conviene saber que el resultado
> ya no corresponde a la definición anterior, y que enmascara errores de forma que de otro modo
> serían detectados.

In [ ]:
lote = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0],
                 [7.0, 8.0, 9.0]])       # 3 muestras x 3 caracteristicas
sesgo = np.array([100.0, 200.0, 300.0])  # un sesgo por caracteristica

print("lote + sesgo  (el vector se repite en cada fila):\n", lote + sesgo)

# El mismo resultado sin broadcasting, escrito como suma de matrices del mismo tamano:
expandido = np.tile(sesgo, (3, 1))
print("\nsesgo expandido a 3x3:\n", expandido)
print("\ncoincide:", np.allclose(lote + sesgo, lote + expandido))

---

## 3. Multiplicación por un escalar

Multiplicar una matriz por un número $\lambda \in \mathbb{R}$ consiste en multiplicar **cada
entrada** por ese número:

$$(\lambda A)_{ij} = \lambda \, a_{ij}$$

Para $\lambda, \psi \in \mathbb{R}$ y $B, C$ matrices del mismo tamaño:

- **Asociativa:** $(\lambda\psi)C = \lambda(\psi C)$.
- **Distributiva:** $(\lambda + \psi)C = \lambda C + \psi C$ y $\lambda(B + C) = \lambda B + \lambda C$.

In [ ]:
lam, psi = 3.0, -2.0

print("3 * A2 =\n", lam * A2)
print("\nasociativa    (lam*psi)*C == lam*(psi*C):",
      np.allclose((lam * psi) * C2, lam * (psi * C2)))
print("distributiva  (lam+psi)*C == lam*C + psi*C:",
      np.allclose((lam + psi) * C2, lam * C2 + psi * C2))
print("distributiva  lam*(B+C) == lam*B + lam*C  :",
      np.allclose(lam * (B2 + C2), lam * B2 + lam * C2))

# Caso frecuente: normalizar una imagen dividiendo entre un escalar.
pixeles = np.array([[0.0, 128.0, 255.0],
                    [64.0, 200.0, 32.0]])
print("\npixeles / 255 =\n", pixeles / 255)

---

## 4. Multiplicación de matrices

### 4.1 La regla

Es la única de las cuatro operaciones que no actúa entrada por entrada. Cada entrada del resultado
se obtiene combinando una **fila completa** de la primera matriz con una **columna completa** de la
segunda, multiplicando posición a posición y sumando:

$$C = AB, \qquad c_{ij} = \sum_{k=1}^{n} a_{ik}\, b_{kj}$$

**Requisito de tamaños.** Para que $AB$ exista, el número de columnas de $A$ debe coincidir con el
número de filas de $B$:

$$\underbrace{A}_{m \times n} \cdot \underbrace{B}_{n \times k} = \underbrace{C}_{m \times k}$$

Las dimensiones interiores deben coincidir y desaparecen; las exteriores determinan la forma del
resultado. Ésta es la comprobación que conviene hacer antes de escribir cualquier producto.

In [ ]:
def multiplicar(A, B):
    """Producto matricial a partir de la definicion c[i,j] = sum_k a[i,k]*b[k,j]."""
    m, n = A.shape
    n2, k = B.shape
    if n != n2:
        raise ValueError(f"formas incompatibles: {A.shape} por {B.shape}")

    C = np.zeros((m, k))
    for i in range(m):                     # cada fila del resultado
        for j in range(k):                 # cada columna del resultado
            for p in range(n):             # producto punto de fila i con columna j
                C[i, j] += A[i, p] * B[p, j]
    return C


A4 = np.array([[1.0, 2.0],
               [3.0, 4.0]])
B4 = np.array([[2.0, 0.0],
               [1.0, 2.0]])

print("a mano =\n", multiplicar(A4, B4))
print("\nnumpy  =\n", A4 @ B4)
print("\niguales:", np.allclose(multiplicar(A4, B4), A4 @ B4))

# El mismo calculo, entrada por entrada, como en el documento:
for i in range(2):
    for j in range(2):
        print(f"entrada ({i + 1},{j + 1}): fila {i + 1} de A con columna {j + 1} de B = "
              f"{A4[i]} . {B4[:, j]} = {A4[i] @ B4[:, j]}")

In [ ]:
# La regla de las formas, comprobada sin calcular nada:
for forma_a, forma_b in [((4, 3), (3, 5)), ((2, 2), (2, 2)), ((3, 1), (1, 3)), ((5, 2), (5, 2))]:
    X, Y = np.zeros(forma_a), np.zeros(forma_b)
    try:
        print(f"{forma_a} @ {forma_b} -> {(X @ Y).shape}")
    except ValueError:
        print(f"{forma_a} @ {forma_b} -> no esta definido (las dimensiones interiores no coinciden)")

### 4.2 El orden importa

A diferencia del producto de números, el producto de matrices **no es conmutativo**: en general
$AB \neq BA$. Con las mismas $A$ y $B$ de arriba,

$$AB = \begin{bmatrix} 4 & 4 \\ 10 & 8 \end{bmatrix} \neq
\begin{bmatrix} 2 & 4 \\ 7 & 10 \end{bmatrix} = BA .$$

La no conmutatividad no es una anomalía: es la propiedad esencial de la operación. Cada matriz
representa una transformación del espacio, y aplicar dos transformaciones en distinto orden produce
resultados distintos. En una red neuronal esto significa que el orden de las capas forma parte de la
definición del modelo.

In [ ]:
print("A @ B =\n", A4 @ B4)
print("\nB @ A =\n", B4 @ A4)
print("\niguales:", np.allclose(A4 @ B4, B4 @ A4))

# Ni siquiera la forma tiene por que coincidir: si A es (2,3) y B es (3,2),
# entonces AB es (2,2) y BA es (3,3).
P = np.arange(6.0).reshape(2, 3)
Q = np.arange(6.0).reshape(3, 2)
print(f"\n(2,3) @ (3,2) -> {(P @ Q).shape}    (3,2) @ (2,3) -> {(Q @ P).shape}")

In [ ]:
# Dos transformaciones del plano, aplicadas en los dos ordenes posibles.
R = np.array([[0.0, -1.0],      # rotacion de 90 grados
              [1.0, 0.0]])
S = np.array([[3.0, 0.0],       # dilatacion horizontal de factor 3
              [0.0, 1.0]])

cuadrado = np.array([[0, 1, 1, 0, 0],
                     [0, 0, 1, 1, 0]], dtype=float)

fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (Mat, titulo) in zip(ejes, [(R @ S, "R @ S: primero dilatar, luego rotar"),
                                    (S @ R, "S @ R: primero rotar, luego dilatar")]):
    transformado = Mat @ cuadrado
    ax.plot(cuadrado[0], cuadrado[1], "o-", color="tab:blue", label="original")
    ax.fill(cuadrado[0], cuadrado[1], alpha=0.2, color="tab:blue")
    ax.plot(transformado[0], transformado[1], "o-", color="tab:red", label="transformado")
    ax.fill(transformado[0], transformado[1], alpha=0.2, color="tab:red")
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.axhline(0, color="gray", lw=0.8); ax.axvline(0, color="gray", lw=0.8)
    ax.grid(alpha=0.3); ax.set_aspect("equal"); ax.set_title(titulo, fontsize=10); ax.legend()
plt.show()

### 4.3 No es el producto entrada por entrada

Multiplicar las entradas correspondientes de dos matrices produce una operación distinta, llamada
**producto de Hadamard** y denotada $A \odot B$:

$$(A \odot B)_{ij} = a_{ij}\, b_{ij}$$

Está perfectamente definida y tiene usos propios, pero **no** es el producto del álgebra lineal. En
NumPy la distinción es la siguiente:

| Operador | Operación | Resultado |
|---|---|---|
| `A * B` | producto de Hadamard | entrada por entrada, misma forma |
| `A @ B` o `np.matmul(A, B)` | producto matricial | fila por columna, formas $(m,n)(n,k)$ |

Confundirlos es el error silencioso más frecuente al empezar: con matrices cuadradas del mismo
tamaño ninguna de las dos operaciones falla, de modo que el programa se ejecuta con normalidad y
entrega un resultado incorrecto.

In [ ]:
print("A * B  (Hadamard, entrada por entrada) =\n", A4 * B4)
print("\nA @ B  (producto matricial)            =\n", A4 @ B4)
print("\niguales:", np.allclose(A4 * B4, A4 @ B4), " <- operaciones distintas, ambas validas")

# Donde el producto de Hadamard si es la operacion correcta: aplicar una mascara.
activaciones = np.array([[0.8, -1.2, 2.0],
                         [0.5, -0.3, 1.1]])
mascara = np.array([[1.0, 0.0, 1.0],
                    [0.0, 1.0, 1.0]])        # dropout: apagar neuronas
print("\nactivaciones * mascara =\n", activaciones * mascara)

### 4.4 Propiedades

Para matrices de formas compatibles:

- **Asociativa:** $(AB)C = A(BC)$.
- **Distributiva:** $(A + B)C = AC + BC$ y $A(C + D) = AC + AD$.
- **Elemento neutro:** $I_m A = A I_n = A$ para $A \in \mathbb{R}^{m \times n}$.

La conmutativa está ausente de la lista deliberadamente. Conviene notar que la asociatividad, pese a
no alterar el resultado, sí altera el **costo**: multiplicar $(AB)C$ y $A(BC)$ requiere números de
operaciones muy distintos cuando las formas son desiguales, y elegir bien el orden es una
optimización real en implementaciones de modelos grandes.

In [ ]:
D4 = np.array([[1.0, 2.0], [0.0, 1.0]])

print("asociativa    (A@B)@C == A@(B@C):", np.allclose((A4 @ B4) @ C2, A4 @ (B4 @ C2)))
print("distributiva  (A+B)@C == A@C + B@C:", np.allclose((A4 + B4) @ C2, A4 @ C2 + B4 @ C2))
print("distributiva  A@(C+D) == A@C + A@D:", np.allclose(A4 @ (C2 + D4), A4 @ C2 + A4 @ D4))

# La asociatividad no cambia el resultado, pero si el numero de operaciones.
F = np.zeros((1000, 5)); G = np.zeros((5, 1000)); H = np.zeros((1000, 2))
costo_izq = 1000 * 5 * 1000 + 1000 * 1000 * 2      # (FG)H
costo_der = 5 * 1000 * 2 + 1000 * 5 * 2            # F(GH)
print(f"\n(F@G)@H : {costo_izq:>12,} multiplicaciones")
print(f"F@(G@H) : {costo_der:>12,} multiplicaciones  <- el mismo resultado, {costo_izq // costo_der}x mas barato")
print("resultados iguales:", np.allclose((F @ G) @ H, F @ (G @ H)))

### Lectura en aprendizaje automático

El producto matricial es la operación dominante en el cómputo de una red neuronal. Una capa densa
evalúa $Y = XW + \mathbf{b}$, donde $X$ contiene una fila por muestra y $W$ una columna por neurona
de salida; el resto de la arquitectura —convoluciones, atención, recurrencia— se reduce, en la
implementación, a productos matriciales de formas particulares.

El motivo por el que ese cómputo se ejecuta en GPU está en el conteo de operaciones: el producto de
una matriz $m \times n$ por una $n \times k$ requiere $mnk$ multiplicaciones y otras tantas sumas.
Para matrices de $1000 \times 1000$ son $10^9$ multiplicaciones en un solo producto. Lo relevante es
que todas las entradas del resultado son **independientes entre sí** y pueden calcularse en
paralelo, lo cual corresponde exactamente al modelo de ejecución de una GPU. La misma razón explica
la diferencia de tiempos de la celda siguiente: `@` delega en una biblioteca BLAS compilada y
optimizada para la jerarquía de memoria, mientras que los ciclos explícitos se interpretan uno a uno.

In [ ]:
import time

n = 150
P = np.random.default_rng(0).random((n, n))
Q = np.random.default_rng(1).random((n, n))

t0 = time.perf_counter(); lento = multiplicar(P, Q); t_lento = time.perf_counter() - t0
t0 = time.perf_counter(); rapido = P @ Q;            t_rapido = time.perf_counter() - t0

print(f"multiplicaciones necesarias: {n ** 3:,}")
print(f"ciclos de Python : {t_lento:.4f} s")
print(f"@ de NumPy (BLAS): {t_rapido:.6f} s")
print(f"factor           : ~{t_lento / max(t_rapido, 1e-9):,.0f}x")
print("mismo resultado  :", np.allclose(lento, rapido))

---

## 5. La matriz identidad

La matriz identidad $I_n \in \mathbb{R}^{n \times n}$ tiene unos en la diagonal y ceros en el resto:

$$I_2 = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}$$

Es el elemento neutro del producto matricial: $I_m A = A I_n = A$. Obsérvese que, a diferencia de lo
que ocurre con el producto en general, la identidad **sí conmuta** con cualquier matriz cuadrada del
mismo tamaño.

In [ ]:
I2 = np.eye(2)
A5 = np.array([[3.0, 5.0],
               [2.0, 7.0]])

print("I2 =\n", I2)
print("\nI2 @ A == A:", np.allclose(I2 @ A5, A5))
print("A @ I2 == A:", np.allclose(A5 @ I2, A5))

# Con una matriz rectangular hacen falta dos identidades de tamanos distintos.
print("\nA (2,3):", A.shape, "  I2 @ A:", (np.eye(2) @ A).shape, "  A @ I3:", (A @ np.eye(3)).shape)

### Lectura en aprendizaje automático

Las redes muy profundas emplean **conexiones residuales**: en lugar de que cada bloque transforme
por completo su entrada, calcula $\mathbf{y} = \mathbf{x} + F(\mathbf{x})$. El término $\mathbf{x}$
equivale a una multiplicación implícita por la identidad, y su efecto es que la transformación
identidad sea el comportamiento por defecto del bloque, de modo que aprender a *no* modificar la
entrada no exige ajuste alguno. Esta construcción, introducida en las redes residuales, es la que
hace entrenables arquitecturas de cientos de capas, y aparece también en cada bloque de un
Transformer.

In [ ]:
def bloque_residual(x, W, b):
    """y = x + F(x), con F una capa densa con activacion ReLU."""
    return x + np.maximum(0.0, x @ W + b)


rng = np.random.default_rng(0)
x = rng.normal(size=(4, 3))
W = rng.normal(size=(3, 3)) * 0.1
b = np.zeros(3)

print("entrada:\n", x)
print("\nsalida del bloque residual:\n", bloque_residual(x, W, b))

# Con pesos nulos el bloque se reduce exactamente a la identidad.
print("\ncon W = 0 el bloque es la identidad:",
      np.allclose(bloque_residual(x, np.zeros((3, 3)), b), x))

---

## 6. La transpuesta

Transponer una matriz consiste en intercambiar sus filas por sus columnas. Se denota $A^\top$ y se
define por $(A^\top)_{ij} = a_{ji}$; si $A$ es de tamaño $m \times n$, entonces $A^\top$ es
$n \times m$.

$$A = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{bmatrix} \;\Longrightarrow\;
A^\top = \begin{bmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{bmatrix}$$

**Propiedades.**

- $(A^\top)^\top = A$.
- $(A + B)^\top = A^\top + B^\top$.
- $(AB)^\top = B^\top A^\top$ — **el orden se invierte**.

La última merece atención: la transpuesta de un producto no es el producto de las transpuestas. La
razón es de formas antes que de valores; si $A$ es $m \times n$ y $B$ es $n \times k$, entonces
$A^\top B^\top$ ni siquiera está definida salvo que $m = k$, mientras que $B^\top A^\top$ lo está
siempre y tiene la forma $k \times m$ que corresponde a $(AB)^\top$.

In [ ]:
print("A =\n", A)
print("\nA.T =\n", A.T, "\nforma:", A.T.shape)

print("\n(A.T).T == A        :", np.allclose(A.T.T, A))
print("(A+B).T == A.T + B.T:", np.allclose((A4 + B4).T, A4.T + B4.T))
print("(A@B).T == B.T @ A.T:", np.allclose((A4 @ B4).T, B4.T @ A4.T))

print("\n(A@B).T =\n", (A4 @ B4).T)
print("\nA.T @ B.T =\n", A4.T @ B4.T, "\n<- distinto: el orden importa tambien al transponer")

# Con matrices rectangulares, la version equivocada ni siquiera esta definida.
M23 = np.arange(6.0).reshape(2, 3)
N34 = np.arange(12.0).reshape(3, 4)
print(f"\n(M@N) es {(M23 @ N34).shape}, luego (M@N).T es {(M23 @ N34).T.shape}")
print(f"N.T @ M.T es {(N34.T @ M23.T).shape}  <- la forma correcta")
try:
    M23.T @ N34.T                # (3,2) @ (4,3): no esta definido
except ValueError as e:
    print("M.T @ N.T ->", e)

### 6.1 Matrices simétricas

Una matriz cuadrada es **simétrica** si coincide con su propia transpuesta, $A = A^\top$; es decir,
si $a_{ij} = a_{ji}$ para todo par de índices.

$$A = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$$

Las matrices simétricas tienen propiedades espectrales notables —autovalores reales y autovectores
ortogonales— que se tratan en el notebook 04. Conviene registrar desde ahora una construcción que
las produce sistemáticamente: para cualquier matriz $X$, los productos $X^\top X$ y $X X^\top$ son
simétricos. Ésa es la razón por la que la simetría aparece en todas partes en estadística y
aprendizaje automático: la matriz de covarianza y la matriz de Gram tienen precisamente esa forma.

In [ ]:
S1 = np.array([[2.0, 1.0],
               [1.0, 2.0]])
print("S es simetrica:", np.allclose(S1, S1.T))

rng = np.random.default_rng(3)
X = rng.normal(size=(6, 3))

print("\nX es (6,3); X.T @ X es", (X.T @ X).shape, "y X @ X.T es", (X @ X.T).shape)
print("X.T @ X simetrica:", np.allclose(X.T @ X, (X.T @ X).T))
print("X @ X.T simetrica:", np.allclose(X @ X.T, (X @ X.T).T))

# La matriz de covarianza de unos datos centrados es de esa forma.
Xc = X - X.mean(axis=0)
cov = (Xc.T @ Xc) / (len(Xc) - 1)
print("\ncovarianza =\n", cov)
print("simetrica:", np.allclose(cov, cov.T), "  coincide con np.cov:", np.allclose(cov, np.cov(X.T)))

### Lectura en aprendizaje automático

La transpuesta es el objeto que aparece cuando se invierte el sentido de una transformación lineal
en el cálculo de derivadas. Si una capa calcula $\mathbf{y} = W\mathbf{x}$ y $L$ es la función de
pérdida, la regla de la cadena da

$$\frac{\partial L}{\partial \mathbf{x}} = W^\top \frac{\partial L}{\partial \mathbf{y}},$$

de modo que la propagación hacia adelante usa $W$ y la propagación hacia atrás usa $W^\top$. No es
una coincidencia de notación: la transpuesta es la representación matricial de la aplicación
adjunta, y por eso el gradiente viaja hacia atrás por la misma red con las matrices transpuestas.

La transpuesta aparece también en la solución de mínimos cuadrados,
$\boldsymbol{\theta} = (X^\top X)^{-1} X^\top \mathbf{y}$, donde $X^\top X$ es la matriz simétrica de
la sección anterior.

In [ ]:
# Las formas de la propagacion hacia adelante y hacia atras son simetricas entre si.
W = rng.normal(size=(4, 3))       # 4 entradas -> 3 salidas
x = rng.normal(size=4)

y = W.T @ x                       # capa: (3,4) @ (4,) -> (3,)
grad_y = rng.normal(size=3)       # gradiente que llega de la capa siguiente
grad_x = W @ grad_y               # hacia atras: (4,3) @ (3,) -> (4,)

print("W:", W.shape, " x:", x.shape, " y:", y.shape)
print("grad_y:", grad_y.shape, " grad_x:", grad_x.shape, " <- la forma de x, como debe ser")

# El gradiente respecto a los pesos tiene la forma de W.
grad_W = np.outer(x, grad_y)
print("grad_W:", grad_W.shape, " igual que W:", grad_W.shape == W.shape)

### Lectura en aprendizaje automático: la suma de matrices

La operación más simple del documento es también la base del entrenamiento distribuido. Como las
matrices de pesos del mismo tamaño se pueden sumar y promediar, tiene sentido la expresión

$$W_{\text{prom}} = \frac{1}{k}\left(W_1 + W_2 + \dots + W_k\right),$$

que es exactamente lo que hace el **aprendizaje federado** —promediar los pesos entrenados en
distintos dispositivos sin centralizar los datos— y lo que hacen las *model soups*, que promedian
varios modelos ajustados con distintas configuraciones. El descenso de gradiente por lotes es el
mismo cálculo aplicado a los gradientes en lugar de a los pesos.

In [ ]:
# Tres replicas del mismo modelo, entrenadas por separado.
rng = np.random.default_rng(7)
W_base = rng.normal(size=(3, 3))
replicas = [W_base + 0.1 * rng.normal(size=(3, 3)) for _ in range(3)]

W_prom = sum(replicas) / len(replicas)

print("W promedio =\n", W_prom)
print("\ndistancia de cada replica al modelo base:",
      [round(float(np.linalg.norm(W - W_base)), 4) for W in replicas])
print("distancia del promedio al modelo base :", round(float(np.linalg.norm(W_prom - W_base)), 4),
      " <- promediar reduce la dispersion")

---

## Tu turno

Los ejercicios son los propuestos en el documento `Matematicas_para_IA_00.pdf`. Sus respuestas están
en el PDF; el objetivo aquí es **calcularlas a mano primero y verificarlas después con código**.

**1.** Calcular $A + B$ con
$A = \begin{bmatrix} 2 & -1 \\ 0 & 3 \end{bmatrix}$,
$B = \begin{bmatrix} 1 & 4 \\ 5 & -2 \end{bmatrix}$.

**2.** Calcular $AB$ con
$A = \begin{bmatrix} 2 & 1 \\ 0 & 3 \end{bmatrix}$,
$B = \begin{bmatrix} 1 & 2 \\ 4 & 0 \end{bmatrix}$, entrada por entrada. Verificar después con `@` y
con la función `multiplicar` definida arriba. Calcular también $BA$ y comprobar que difieren.

**3.** Obtener la transpuesta de
$A = \begin{bmatrix} 5 & -2 \\ 0 & 1 \\ 3 & 4 \end{bmatrix}$ e indicar la forma de $A$ y de
$A^\top$.

**4.** ¿Es $A = \begin{bmatrix} 1 & 3 \\ 3 & 1 \end{bmatrix}$ simétrica? Escribir una función
`es_simetrica(A)` que lo decida para una matriz cualquiera, contemplando el caso de una matriz no
cuadrada.

**5.** Sin ejecutar nada, indicar la forma del resultado de cada producto o si no está definido:
$(4\times3)(3\times5)$, $(2\times2)(2\times2)$, $(3\times1)(1\times3)$, $(5\times2)(5\times2)$.
Verificar después con `np.zeros` y `.shape`.

**6.** Comprobar con código que $(ABC)^\top = C^\top B^\top A^\top$ para tres matrices de formas
compatibles, y explicar por qué el orden se invierte por completo.

**7.** Construir dos matrices $2 \times 2$ que **sí** conmuten ($AB = BA$) y en las que ninguna sea
la identidad ni un múltiplo de ella. Sugerencia: considerar potencias de una misma matriz.

**8.** Verificar que para una matriz $X$ cualquiera, $X^\top X$ es simétrica, y comprobar que
$X^\top X$ y $X X^\top$ tienen formas distintas cuando $X$ no es cuadrada.

In [ ]:
# Tu codigo aqui.
A = np.array([[2.0, -1.0],
              [0.0, 3.0]])
B = np.array([[1.0, 4.0],
              [5.0, -2.0]])

# 1.

---

## Resumen

| Operación | Notación | En NumPy | Condición | Resultado |
|---|---|---|---|---|
| Suma | $A + B$ | `A + B` | misma forma | misma forma |
| Producto por escalar | $\lambda A$ | `lam * A` | siempre | misma forma |
| Producto matricial | $AB$ | `A @ B` | $(m,n)(n,k)$ | $(m,k)$ |
| Producto de Hadamard | $A \odot B$ | `A * B` | misma forma | misma forma |
| Transpuesta | $A^\top$ | `A.T` | siempre | $(n,m)$ |
| Identidad | $I_n$ | `np.eye(n)` | cuadrada | $I_nA = AI_n = A$ |

| Propiedad | ¿Se cumple? |
|---|---|
| $A + B = B + A$ | Sí |
| $AB = BA$ | **No**, en general |
| $(AB)C = A(BC)$ | Sí |
| $A(B + C) = AB + AC$ | Sí |
| $(AB)^\top = B^\top A^\top$ | Sí (el orden se invierte) |
| $(AB)^\top = A^\top B^\top$ | **No** |

Tres ideas para retener:

1. **La forma manda.** Antes de calcular, conviene verificar la compatibilidad de formas y anticipar
   la del resultado; la mayoría de los errores se detectan ahí.
2. **`*` no es `@`.** El asterisco es el producto de Hadamard y el arroba el producto matricial.
   Ambos son válidos, ninguno falla con matrices cuadradas del mismo tamaño, y sólo uno es el que
   se quería.
3. **El orden no es conmutativo, pero sí asociativo.** $AB \neq BA$ altera el resultado;
   $(AB)C = A(BC)$ no lo altera, pero sí el costo del cálculo.

## Qué sigue

- La versión en script, más corta y sin explicaciones: [`00_propiedades_matrices.py`](00_propiedades_matrices.py)
- El documento de la sesión: `Matematicas_para_IA_00.pdf`
- El siguiente notebook: [`01 · Vectores y espacios vectoriales`](01_vectores_y_espacios_vectoriales.ipynb)
- Los demás módulos, en el [README del repositorio](../../README.md)

**Referencia.** Deisenroth, M. P., Faisal, A. A. y Ong, C. S. (2020). *Mathematics for Machine
Learning*. Cambridge University Press, sección 2.2.

---

*Material abierto bajo licencia MIT. ¿Encontraste un error o quieres aportar un ejercicio?
Lee [CONTRIBUTING.md](../../CONTRIBUTING.md).*